# Beetles: Directional response to a visual stimulus

A worked example of modelling a **circular outcome** (a heading/direction in radians) with a **categorical stimulus**, using `bambi`'s `VonMises` family together with `unwrap`'s circular link functions.

**What is modelled**

- The *mean direction* $\mu$ of the beetle's heading, regressed on a `Light` (treatment) factor.
- The concentration $\kappa > 0$ of the **within-individual** distribution of headings - how tightly an *individual's own* trial headings cluster around that individual's own mean direction $\mu$ - also regressed on `Light`, with a random effect for `Individual` in one of the variants.
- The **between-individual** concentration $\kappa_\mu$ - the concentration of the individuals' mean directions *across the population* - is *not* a model parameter: it is estimated post-hoc from the individuals' own $\mu$ estimates. The **change-in-dispersion** ($\kappa_\mu$) test asks whether the stimulus changes how consistent the individuals' own heading biases are with one another.

**Analysis flow**

1. **Import `unwrap`** and load the per-individual directional data with its treatment labels.
2. **Fit three nested von Mises models** on the mean direction:
   (a) a treatment effect on the hyperparameters (random `Individual` on $\kappa$),
   (b) a no-change-in-$\kappa$ variant, and
   (c) a no-population-$\mu$ variant.
3. **Compare models** with leave-one-out (LOO) cross-validation / ELPD to decide which hypothesis the data support.
4. **Posterior predictive checks** to validate that the fitted models reproduce the directional distribution of the data.

All directional quantities are handled on the circle: `unwrap`'s `VonMises` prior and link functions keep $\mu \in \mathbb{R}/2\pi\mathbb{Z}$ and $\kappa > 0$ inside the differentiable PyTensor graph, so `bambi` can fit them as standard GLM terms.


## Setup

From the repository root, in a terminal:

```bash
uv venv --python 3.12
uv pip install -e ".[notebook]"
uv run --no-project jupyter lab notebooks/beetles.ipynb
```

Select the kernel from this virtual environment and run all cells in order.
The `.[notebook]` extra installs the modelling stack plus everything this
notebook imports (`seaborn`, `pyarrow`, ...).


In [ ]:
%matplotlib inline
from importlib.metadata import version
import os
import sys

import numpy as np
import arviz as az
import matplotlib.pyplot as plt

import unwrap
from unwrap.link.numpy import circular_modulo
from unwrap.circular import circular_summary

# The data, post-hoc and figure code lives in sibling scripts next to this
# notebook (beetles_preprocess.py, beetles_posthoc.py, beetles_plots.py);
# make them importable regardless of the kernel's working directory
# (the kernel may run from notebooks/ or from the repository root).
sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(unwrap.__file__)), "notebooks"))

import beetles_preprocess as preprocess
import beetles_posthoc as posthoc
import beetles_plots as plots

SEED = 42

{package: version(package) for package in (
    "unwrap", "bambi", "pymc", "arviz",
    "numpy", "pandas", "scipy", "matplotlib",
)}


## Figure style

All figures share one palette and font: light **OFF** is grey, light **ON** is gold, Arial throughout. Each figure cell renders a low-resolution image inline and saves the 300 dpi vector file it was given through the explicit `save_path=` argument, under `figures/beetles/` (the saved files are gitignored).


In [ ]:
# One palette and figure style for every figure (grey = light OFF, gold = light
# ON, Arial, 300 dpi, type-42 outlines); the matplotlib "how" lives in the
# sibling script.
plots.set_style()


# Read in data


In [ ]:
# Load the raw security-light trials from unwrap-data and build the modelling table
# (see notebooks/beetles_preprocess.py).
df = preprocess.load_beetle_model_table()
df.head()


# Plot data

Polar grid of raw headings per individual, light OFF (grey) vs light ON (gold), with the circular mean (solid) and median (dashed).


In [ ]:
plots.plot_data_polar(df, save_path="figures/beetles/data_individual_polar_grid.pdf")
plt.show()


# Model with a treatment effect on hyperparameters

We fit a von Mises hierarchical model in `bambi` where both the mean direction $\mu$ and the concentration $\kappa$ (the *within-individual* concentration: how tightly each individual's own trial headings cluster around its own mean direction) are regressed on `Light`, with random effects for each `Individual`. `unwrap` provides the circular link functions that keep $\mu$ on the circle and $\kappa > 0$ inside the PyTensor graph. The $\kappa$ random effects are fitted non-centered (each is written as a half-normal $\sigma$ times a standard-normal offset), the reparameterization that removes the divergent transitions the centered model produced.


In [ ]:
# from fastprogress import fastprogress
# fastprogress.printing = lambda: True

import numpy as np
import pandas as pd
import bambi as bmb
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

from unwrap.link.bambi import LINKS
from unwrap.link.numpy import inverse_softplus
from unwrap.prior import CircularUniformPrior

# Define the model formula with updated, capitalized variable names.
formula_treat = bmb.Formula(
    "HeadingRadians ~ Light + (1 + Light|Individual)",
    "kappa ~ Light + (1 + Light|Individual)",
)

# Specify priors for the model with capitalized keys and updated variable names.
priors_treat= {
    # Fixed effects for μ:
    "Intercept": bmb.Prior("VonMises", mu=0, kappa= 1e-6),
    "Light": bmb.Prior("VonMises", mu=0, kappa=2.0),#von Mises
    # Individual-level effects for μ:
    "1|Individual": bmb.Prior(
        "VonMises", mu=0, kappa=bmb.Prior("LogNormal", mu=np.log(30), sigma=0.5) # high shrinkage prior
    ),
    "Light|Individual": bmb.Prior(
        "VonMises", mu=0, kappa=bmb.Prior("LogNormal", mu=np.log(30), sigma=0.5) # potential for large changes 
    ),
    # Priors for the κ-model (inside the "kappa" dictionary):
    "kappa": {
        # Fixed effects for κ:
        "Intercept": bmb.Prior("Normal", mu=inverse_softplus(5.0), sigma=1.0), #wider possible range
        "Light": bmb.Prior("Normal", mu=0, sigma=1.0),#slightly wider
        # Individual-level effects for κ (non-centered; the sigma * offset
        # reparameterization removes the divergent transitions of the
        # centered fit):
        "1|Individual": bmb.Prior(
            "Normal", mu=0, sigma=bmb.Prior("HalfNormal", sigma=1.0), noncentered=True
        ),
        "Light|Individual": bmb.Prior(
            "Normal", mu=0, sigma=bmb.Prior("HalfNormal", sigma=1.0), noncentered=True
        ),
    },
}

# Build the model using the updated formula and priors.
model_treat = bmb.Model(
    formula_treat,
    df,
    family="vonmises",
    priors=priors_treat,
    link={"mu": LINKS["circular_modulo"], "kappa": LINKS["softplus"]},
    noncentered=False,
    center_predictors=False,
    dropna=True,
)

print(model_treat)

# Fit the model (seeded for reproducibility).
idata_treat = model_treat.fit(
    tune=2000, draws=2000, target_accept=0.9,
    random_seed=SEED, progressbar=False,
)

# Store the per-observation log-likelihood that the post-hoc cross-check
# validates against (the pymc 6 replacement for
# idata_kwargs={"log_likelihood": True}; posthoc.graph_free_rvs supplies
# the non-centered *_offset free RVs that bambi does not store).
idata_treat = pm.compute_log_likelihood(
    posthoc.graph_free_rvs(idata_treat, model_treat),
    model=model_treat.backend.model,
    progressbar=False,
)

# Report the divergence count first: R-hat is only meaningful when it is 0.
print("Divergences:", int(idata_treat.sample_stats.diverging.sum()))


## Inspect summary

Posterior summaries of the fixed effects and the individual-level effects. The fit cell above reports the divergence count first: while divergent transitions exist, $\hat{R}$ and the ESS columns would be meaningless, so those values below are only interpretable because that count is 0.


In [ ]:
circular_summary(
    idata_treat, 
    var_names=["Intercept", "Light", "kappa_Intercept", "kappa_Light", "1|Individual_kappa", "Light|Individual_kappa",
                "kappa_1|Individual_sigma", "kappa_Light|Individual_sigma"],
    circ_var_names=["Intercept", "Light"]
)


In [ ]:
circular_summary(
    idata_treat, 
    var_names=["1|Individual", "Light|Individual",
                "kappa_1|Individual", "kappa_Light|Individual"],
    circ_var_names=["1|Individual", "Light|Individual"]
)


## Visualising the treatment model

Panel **b** (left): a polar kernel density of the population-level posterior draws of $\mu$ under each lighting condition, with the floodlight direction marked as an arc. Panels **c**-**f** (right): **c** - the *within-individual* concentration $\kappa$, as the posterior mean resultant length $\mathrm{A}(\kappa)$ of an individual's own headings around its own mean direction, off vs on; **d** - the *between-individual* concentration $\kappa_\mu$, as the mean resultant length of the individuals' mean directions around the population mean (per draw), off vs on; **e** - the posterior predictive probability mass within $\pm45^{\circ}$ of the floodlight; **f** - the change in that mass with the light switched on.


In [ ]:
# Floodlight direction, in radians (-135 degrees).
plots.plot_polar_mu_kde(
    idata_treat, df, circular_modulo,
    flood_dir=np.deg2rad(-135),
    kappa_bw=80.0,
    save_path="figures/beetles/treat_polar_mu_kde_clean.pdf",
)
plt.show()


In [ ]:
# Floodlight direction, in radians (-135 degrees).
from unwrap.link.numpy import softplus

FLOOD_DIR = np.deg2rad(-135)
plots.plot_concentration(
    idata_treat, df, circular_modulo, softplus,
    flood_dir=FLOOD_DIR,
    hdi_prob=0.89,
    save_path="figures/beetles/treat_concentration_plus_floodlight.pdf",
)
plt.show()


# Inspect the change in $\kappa_\mu$

The paper's main hypothesis test is whether the *between-individual* concentration $\kappa_\mu$ - the concentration of the individuals' mean directions across the population, i.e. how tightly the individuals' own heading biases cluster (in contrast to the *within-individual* $\kappa$, which describes each individual's own trial-to-trial scatter) - changes with the stimulus. We estimate $\kappa_\mu$ post-hoc two ways: from the posterior point estimates of each individual's $\mu$, and per-draw.


## Extract individual $\mu$ estimates and calculate mean $\kappa_\mu$


In [ ]:
# Posterior distribution of the individual mean directions, and the between-individual concentration of that distribution (kappa_mu), off and on -- see notebooks/beetles_posthoc.py.
kappamu_individual = posthoc.kappamu_individual(idata_treat, df)


## Extract $\kappa_\mu$ estimates for each draw


In [ ]:
kappamu_posterior_idata = posthoc.kappamu_per_draw(idata_treat, df)

az.summary(kappamu_posterior_idata['id_data'], ci_prob=0.95, ci_kind="hdi")


We can see here that for nearly all draws the difference in (post-hoc estimated) $\kappa_\mu$ between the two conditions is positive and large. Calculating $\kappa_\mu$ for the lights off (intercept) condition we get a lower value than $\kappa_\mu$ estimated from the model (`softplus('1|Individual_kappa')`), but it is consistent with the distribution of individual heading biases (`1|Individual`) estimated using those values of $\kappa_\mu$. In a model written using `pymc` directly, we could reformulate the model to estimate both `kappa_mu_intercept` and `kappa_mu_delta` explicitly.


## Posterior predictive checks

A supplementary posterior predictive check compares a nonparametric bootstrap of the *data* against the *model* posterior for the group mean direction, group alignment, and per-individual mean direction and concentration, in both lighting conditions. The bootstrap resamples individuals (and, within each individual, the per-condition headings); the model side evaluates the same statistics from the posterior draws of every observation's response parameters, cross-checked against the stored log-likelihood. The statistics and the figure both live behind the single `plots.plot_ppc` call below (statistics in notebooks/beetles_posthoc.py, the figure in notebooks/beetles_plots.py).


In [ ]:
# Everything (bootstrap resampling, posterior statistics, log-likelihood
# cross-check, and the figure) is inside the call below.
plots.plot_ppc(idata_treat, model_treat, df,
               save_path="figures/beetles/beetle_posterior_predictive_check.pdf")
plt.show()
